# Rule: **build_industrial_energy_demand_per_country_today**


**Description**

This rule builds industrial energy demand at country level using energy balances from JRC-IDEES and physical production (kton/a). Annual demand is disaggregated by sector and energy carrier, with results expressed in TWh/a. No time series are available. For countries outside the EU28, energy demand is estimated using an average intensity based on production volume.

The configuration parameters that determine present industrial energy demands are defined under the **industry** and **sector** section of the config file:  
- industry.MWh_H2_per_tCl
- industry.MWh_elec_per_tCl
- industry.MWh_CH4_per_tMeOH
- industry.MWh_elec_per_tMeOH
- industry.MWh_H2_per_tNH3_electrolysis
- industry.MWh_elec_per_tNH3_electrolysis
- industry.MWh_NH3_per_tNH3
- industry.reference_year
- sector.ammonia

**Inputs**

- resources/{prefix}/{name}/`industrial_production_per_country.csv`
- resources/{prefix}/{name}/`transformation_output_coke.csv`
- data/jrc_idees/archive/{database date}/{country}/`JRC-IDEES-{year}_EnergyBalance_{country}.xlsx`

Note: "archive" directories and database years may vary with different versions of PyPSA-EUR/PyPSA-Spain.

**Outputs**

- resources/{prefix}/{name}/`industrial_energy_demand_per_country_today.csv`

In [ ]:
######################################## Parameters

### Run
prefix = ''
name = ''

In [ ]:
##### Imports
import os 
import sys
import pandas as pd
import plotly.graph_objects as go


##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp

##### Read params.yaml
params = xp.read_params('../params.yaml')

##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

## `industrial_energy_demand_per_country_today.csv`  
Load the file and preview its content.

In [ ]:
file = f"industrial_energy_demand_per_country_today.csv"

ind_energy_today = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
    header=0,
    index_col=0,
)
ind_energy_today.index = ind_energy_today.index.astype(str)
ind_energy_today.index.values[0] = "Sector → , Energy carrier ↓ "
ind_energy_today

See today's industrial energy demand for a specific country

In [ ]:
# Select the country
country_code = "ES"

# Filter by country
cols = [c for c in ind_energy_today.columns if str(c).startswith(country_code)]

ind_energy_today_country = ind_energy_today[cols]

# Header (sectors)
ind_energy_today_country.rows = ind_energy_today_country.iloc[0]

# Index (energy carriers)
ind_energy_today_country.columns = ind_energy_today_country.iloc[0]

ind_energy_today_country = ind_energy_today_country.iloc[2:] # NaN values removed

ind_energy_today_country

Select energy carriers and sectors to plot

In [ ]:
#################### Parameters

### Industrial sectors to plot. Choose any subset, in desired order.
sectors_to_plot = [
    "Aluminium - primary production",
    "Aluminium - secondary production",
    "Alumina production",
    "Ammonia",
    "Cement",
    "Ceramics & other NMM",
    "Chlorine",
    "Electric arc",
    "Food, beverages and tobacco",
    "Glass production",
    "HVC",
    "Integrated steelworks",
    "Machinery equipment",
    "Methanol",
    "Other chemicals",
    "Other industrial sectors",
    "Other non-ferrous metals",
    "Paper production",
    "Pharmaceutical products etc.",
    "Printing and media reproduction",
    "Pulp production",
    "Textiles and leather",
    "Transport equipment",
    "Wood and wood products",
]

### Energy carriers to plot. Choose any subset, in desired order.
energy_carriers = [
    "biomass",
    "electricity",
    "gas",
    "heat",
    "liquid",
    "solid",
    "waste",
    "hydrogen",
    "other",
]

############ Sector colours
colours_sectors = {
    "Aluminium - primary production": "#437812",
    "Aluminium - secondary production": "#8ee87b",
    "Alumina production": "#24ffc1",
    "Ammonia": "#ff9244",
    "Cement": "#000000",
    "Ceramics & other NMM": "#b1e0a9",
    "Chlorine": "#ff9896",
    "Electric arc": "#000dff",
    "Food, beverages and tobacco": "#9467bd",
    "Glass production": "#c5b0d5",
    "HVC": "#d62728",
    "Integrated steelworks": "#6ba4ee",
    "Machinery equipment": "#e377c2",
    "Methanol": "#f7b6d2",
    "Other chemicals": "#f728d5",
    "Other industrial sectors": "#BCBCBC",
    "Other non-ferrous metals": "#ffff1e",
    "Paper production": "#7e7e53",
    "Pharmaceutical products etc.": "#17becf",
    "Printing and media reproduction": "#9edae5",
    "Pulp production": "#393b79",
    "Textiles and leather": "#55103C",
    "Transport equipment": "#f5c25c",
    "Wood and wood products": "#9d642f",
}

############ Energy carrier colours
colours_carriers = {
    "biomass": "#2ca02c",
    "electricity": "#1f77b4",
    "gas": "#ff7f0e",
    "heat": "#d62728",
    "liquid": "#9467bd",
    "solid": "#444444",
    "waste": "#bcbd22",
    "hydrogen": "#17becf",
    "other": "#e377c2"
}

See a specific sector

In [ ]:
#################### Parameters

sector = "Alumina production" 

#################### Prepare data

values = [
    ind_energy_today_country.loc[carrier, sector]
    for carrier in energy_carriers
]

df = pd.DataFrame({
    "carrier": energy_carriers,
    "value": pd.to_numeric(values, errors="coerce")
})

#################### Plot

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=df["carrier"],
        y=df["value"],
        marker=dict(color=[colours_carriers.get(c) for c in df["carrier"]])
    )
)

#################### Layout

fig.update_layout(
    height=500,
    width=900,
    title=f"Present energy demand for {sector} ({country_code})",
    title_x=0.5,

    xaxis_title="Energy carrier",
    yaxis_title="Energy demand (TWh/a)",

    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.show()

How is energy demand distributed across the industry?


In [ ]:
#################### Prepare data

data = []

for sector in sectors_to_plot:
    for carrier in energy_carriers:
        value = ind_energy_today_country.loc[carrier, sector]
        data.append({
            "sector": sector,
            "carrier": carrier,
            "value": value
        })

df_long = pd.DataFrame(data)
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")

#################### Plot

fig = go.Figure()

for carrier in energy_carriers:

    df_c = df_long[df_long["carrier"] == carrier]

    fig.add_trace(
        go.Bar(
            ##### Vertical bars
            x=df_c["sector"],
            y=df_c["value"],
            orientation="v",
            ##### Horizontal bars
            # y=df_c["sector"],
            # x=df_c["value"],
            # orientation="h",
            name=carrier,
            marker=dict(color=colours_carriers.get(carrier)),
        )
    )

#################### Layout

fig.update_layout(
    barmode="stack",
    height=600,
    width=1100,
    title=f"Present energy demand by industrial sector ({country_code})",
    title_x=0.5,

    xaxis_title="Annual energy demand (TWh/a)",
    yaxis_title="Industrial sector",
    font=dict(size=14),

    legend=dict(
        traceorder="normal",
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02,
        title="Energy carrier"
    ),

    margin=dict(l=120, r=200, t=80, b=40),
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.update_yaxes(
    categoryorder="array",
    categoryarray=sectors_to_plot[::-1]
)

fig.show()

How is energy demand distributed for each energy carrier?

In [ ]:
#################### Prepare data
data = []

for carrier in energy_carriers:
    for sector in sectors_to_plot:

        if sector in ind_energy_today_country.columns:
            value = ind_energy_today_country.loc[carrier, sector]

            data.append({
                "carrier": carrier,
                "sector": sector,
                "value": value
            })

df_long = pd.DataFrame(data)

df_long["carrier"] = pd.Categorical(df_long["carrier"], ordered=True)
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")

#################### Plot

fig = go.Figure()

for sector in sectors_to_plot:

    df_s = df_long[df_long["sector"] == sector]

    fig.add_trace(
        go.Bar(
            ##### Vertical bars
            x=df_s["carrier"],
            y=df_s["value"],
            orientation="v",
            ##### Horizontal bars
            #y=df_s["carrier"],
            #x=df_s["value"],
            # orientation="h",
            name=sector,
            marker=dict(color=colours_sectors.get(sector)),
        )
    )

#################### Layout

fig.update_layout(
    barmode="stack",
    height=650,
    width=1200,
    title=f"Present energy demand by carrier ({country_code})",
    title_x=0.5,

    xaxis_title="Annual energy demand (TWh/a)",
    yaxis_title="Energy carrier",
    font=dict(size=14),
    legend=dict(
        traceorder="normal",
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02,
        title="Industrial sector"
    ),
    margin=dict(l=120, r=250, t=80, b=40),
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.update_yaxes(
    categoryorder="array",
    categoryarray=energy_carriers[::-1]
)
fig.show()
